### 02_enriched_utils : Tasks 2 Library

Enriched-layer utilities: `clean_name`, `clean_phone`, `load_enriched_cust_data`, `load_enriched_product_data`, `clean_raw_orders_data`, `load_enriched_orders_data`. Imported by `02_data_enrichment` via `%run`; its code cells are executed directly by the test suite (`tests/conftest.py`).

In [0]:
import pyspark.sql.functions as F

#### Tasks 2 : Enriched Data Loads

Run in `02_data_enrichment`.

Task 2 : Cleaning Customers Data - `clean_name`, `clean_phone` and `load_enriched_cust_data`

In [0]:
def clean_name(col_name="customer_name"):
    """
    Returns a Column expression that cleans a name column by:
      1. Removing junk runs : special characters, underscores, and digits :
         together with the spaces around them
      2. Removing apostrophes followed by a lowercase letter (corruption like
         "Kristi'na"), while keeping genuine ones like "O'Rourke"
      3. Collapsing leftover multiple spaces into a single space and trimming
      4. Joining words across a space between two lowercase letters
         ("Kat rina" -> "Katrina") and stripping leftover punctuation
    """
    # 1. drop junk runs (special chars, underscores, digits) and the spaces around them
    cleaned = F.regexp_replace(F.col(col_name), r"\s*[^\p{L}'\s]+(?:\s+[^\p{L}'\s]+)*", "")
    # remove apostrophe when the next letter is lowercase
    cleaned = F.regexp_replace(cleaned, r"(\p{L})\s*'(\p{Ll})", "$1$2")
    # 3. collapse runs of spaces into a single space and trim
    cleaned = F.trim(F.regexp_replace(cleaned, r"\s+", " "))
    # remove space between two lowercase letters
    cleaned = F.regexp_replace(cleaned, r"(\p{Ll})\s+(\p{Ll})", "$1$2")
    # strip leftover punctuation but KEEP apostrophes inside names (O'Rourke)
    cleaned = F.regexp_replace(cleaned, r"[^\p{L}\s']", "")
    # drop double quotes from the names
    return F.regexp_replace(cleaned, r"^['\"]+|['\"]+$", "")

def clean_phone(col_name="phone"):
    """
    Returns a Column expression that cleans a phone number column by:
      1. Removing the "x" extension and anything after it
      2. Removing all special characters (only digits remain)
      3. Removing the leading "001" country code if present
      4. Keeping the number only if it is exactly 10 digits, otherwise NULL
    """
    # 1. cut off any "x123"-style extension and everything after it
    no_ext = F.regexp_replace(F.col(col_name), r"[xX]\d+.*$", "")
    # 2. strip everything that is not a digit
    digits = F.regexp_replace(no_ext, r"[^0-9]", "")
    # 3. drop a leading "001" country code
    no_cc = F.regexp_replace(digits, r"^001", "")
    # 4. keep only exact 10-digit numbers (prefixed "+1"); everything else becomes NULL
    return F.when(F.length(no_cc) == 10, F.concat(F.lit("+1"), no_cc))

def clean_str_columns(df):
    """
    Returns a DataFrame with every column cast to string and trimmed of
    surrounding whitespace.
    """
    for c, dtype in df.dtypes:
        df = df.withColumn(c, F.trim(F.col(c).cast("string")))
    return df



In [0]:
def load_enriched_cust_data():
    """
    Build enriched.customers from raw.customers.

    Cleans customer names and phone numbers, trims all columns, and keeps a
    single row per customer_id before overwriting enriched.customers
    (schema included).
    """
    enriched_cust_df = (
        spark.table("raw.customers")
            .withColumn("customer_name", clean_name("customer_name")) \
            .withColumn("phone", clean_phone("phone"))
    )
    # trim whitespace on all columns
    enriched_cust_df = clean_str_columns(enriched_cust_df)

    # keep one row per customer
    enriched_cust_df = enriched_cust_df.dropDuplicates(["customer_id"])
    enriched_cust_df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("enriched.customers")

Task 2 : Cleaning Product Data - `load_enriched_product_data`

In [0]:
def load_enriched_product_data():
    """
    Build enriched.products from raw.products.

    Trims string columns, casts price_per_product to decimal(10,2), drops rows
    without a product_id, and keeps a single row per product_id before
    overwriting enriched.products (schema included).
    """
    raw_prod = spark.table("raw.products")

    # trim whitespace on all string columns
    for c, dtype in raw_prod.dtypes:
        if dtype == "string":
            raw_prod = raw_prod.withColumn(c, F.trim(F.col(c)))

    # fixed-precision decimal so later aggregations are exact
    enriched_prod = raw_prod.withColumn("price_per_product", F.expr("cast(price_per_product as decimal(10,2))"))
    # discard rows without a product key, then keep one row per product
    enriched_prod = enriched_prod.filter(F.col("product_id").isNotNull()).dropDuplicates(["product_id"])

    enriched_prod.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("enriched.products")

Task 2 : Cleaning Orders Data - `clean_raw_orders_data` and `load_enriched_orders_data`

In [0]:
def clean_raw_orders_data(orders_raw):
    """
    Clean and deduplicate the raw orders data.

    Parses order/ship dates (day-first formats), casts the numeric columns,
    trims string columns, and keeps only the latest row per (order_id,
    product_id) pair : ranked by order_date then row_id : so duplicate order
    lines collapse to their most recent version.

    Returns the cleaned, deduplicated DataFrame (not persisted).
    """
    from pyspark.sql.window import Window

    enriched_orders_df = (
        orders_raw
        # day-first dates: try single-digit day first, then zero-padded
        .withColumn("order_date", F.coalesce(
            F.try_to_date("order_date", "d/M/yyyy"),
            F.try_to_date("order_date", "dd/MM/yyyy")))
        .withColumn("ship_date", F.coalesce(
            F.try_to_date("ship_date", "d/M/yyyy"),
            F.try_to_date("ship_date", "dd/MM/yyyy")))
        # cast numerics to proper types for downstream calculations
        .withColumn("price", F.col("price").cast("decimal(10,2)"))
        .withColumn("profit", F.col("profit").cast("decimal(10,2)"))
        .withColumn("discount", F.col("discount").cast("decimal(10,2)"))
        .withColumn("quantity", F.col("quantity").cast("int"))
        .withColumn("row_id", F.col("row_id").cast("int"))
    )
    # trim whitespace on all string columns
    for c, dt in enriched_orders_df.dtypes:
        if dt == "string":
            enriched_orders_df = enriched_orders_df.withColumn(c, F.trim(F.col(c)))

    # rank rows within each (order_id, product_id) pair, most recent first
    window = Window.partitionBy("order_id", "product_id").orderBy(F.col("order_date").desc(), F.col("row_id").desc())
    return (
        enriched_orders_df.withColumn("rank", F.row_number().over(window))
        .filter(F.col("rank") == 1)   # keep only the latest version of each order line
        .drop("rank")
        )

def load_enriched_orders_data(enriched_orders_df):
    """
    Persist the cleaned orders DataFrame as enriched.orders.

    Selects the final order columns and overwrites the enriched.orders
    table (schema included).
    """
    (
        enriched_orders_df
        .select("order_id", "customer_id", "product_id", "order_date", "ship_date", 
                "ship_mode", "discount", "quantity", "price", "profit", "row_id")
        .write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("enriched.orders")
    )